# 02 - Collecte, audit & dictionnaire de données
Objectifs:
- Cartographier les sources (raw + enrichissement)
- Data audit: complétude, cohérence, granularité, doublons
- Data dictionary

Entrée: data/processed/dataset_model.csv (généré par le notebook process_data)


In [1]:
import os
import numpy as np
import pandas as pd

DATA_PATH = os.path.join("..","data", "processed", "dataset_model.csv")
df = pd.read_csv(DATA_PATH)

print("Processed shape:", df.shape)
df.head(3)

C:\Users\HL\AppData\Local\Temp\ipykernel_15036\4266000332.py:6: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


Processed shape: (101766, 53)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,diag_1_cat,diag_2_cat,diag_3_cat
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,NO,DIABETES,UNKNOWN,UNKNOWN
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,No,No,No,Ch,Yes,>30,OTHER,DIABETES,OTHER
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,Yes,NO,OTHER,DIABETES,SUPPORT_CARE_V


## 1) Cartographie des sources
 - Source 1 : UCI Diabetes (diabetic_data.csv)
 - Source 2 : Référentiel diag_category.csv (mapping diag -> catégorie)

la source 2 est construite d'apres la classification ICD-9, qui nous a permis de remplacer les nombres dans notre data en diagnostiques separes par categories.


## 2) Audit qualité — complétude / manquants / doublons

In [2]:
# manquants NA
na_rate = df.isna().mean().sort_values(ascending=False)
na_rate.head(15)

weight                0.968585
max_glu_serum         0.947468
A1Cresult             0.832773
medical_specialty     0.490822
payer_code            0.395574
race                  0.022336
diag_3                0.013983
diag_2                0.003518
diag_1                0.000206
age                   0.000000
gender                0.000000
patient_nbr           0.000000
encounter_id          0.000000
num_lab_procedures    0.000000
time_in_hospital      0.000000
dtype: float64

In [3]:
# doublons exacts
dup_exact = df.duplicated().mean()
dup_exact

np.float64(0.0)

In [4]:
# Distribution target
TARGET = "readmitted"
df[TARGET].value_counts(dropna=False)

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

## 3) Cohérence / granularité
 - Vérification des identifiants: encounter_id, patient_nbr.
 - Verification de la validité des données temporelles.
 Conclusion : les données sont suffisamment précises et cohérentes.

In [7]:
print("Nombres de cas de temps invalides : ", len(df[df['time_in_hospital'] < 0]))

for col in ["encounter_id", "patient_nbr"]:
    if col in df.columns:
        print(col, "unique:", df[col].nunique(), "rows:", len(df))

Nombres de cas de temps invalides :  0
encounter_id unique: 101766 rows: 101766
patient_nbr unique: 71518 rows: 101766


## 4) Data dictionary
 Table (colonne, type, % NA, exemples de valeurs)


In [6]:
dictionary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "na_rate": [df[c].isna().mean() for c in df.columns],
    "example_values": [", ".join(map(str, df[c].dropna().unique()[:5])) for c in df.columns]
}).sort_values("na_rate", ascending=False)

dictionary.head(20)

,column,dtype,na_rate,example_values
5,weight,str,0.968585,"[75-100), [50-75), [0-25), [100-125), [25-50)"
22,max_glu_serum,str,0.947468,">300, Norm, >200"
23,A1Cresult,str,0.832773,">7, >8, Norm"
11,medical_specialty,str,0.490822,"Pediatrics-Endocrinology, InternalMedicine, Fa..."
10,payer_code,str,0.395574,"MC, MD, HM, UN, BC"
2,race,str,0.022336,"Caucasian, AfricanAmerican, Other, Asian, Hisp..."
20,diag_3,str,0.013983,"255, V27, 403, 250, V45"
19,diag_2,str,0.003518,"250.01, 250, 250.43, 157, 411"
18,diag_1,str,0.000206,"250.83, 276, 648, 8, 197"
4,age,str,0.000000,"[0-10), [10-20), [20-30), [30-40), [40-50)"


## 5) Gaps data / biais potentiels
    - Certaines variables sont très manquantes, mais nos analyses porteront sur ceux qui sont suffisament présentes.
    - Limite d'analyses causales a cause du manque des données de date.